# 04 — Pull Conflict Event Labels

This notebook pulls the three datasets that provide **ground-truth labels** for the
civil war, coup, and state failure outcome models.

| Dataset | Label(s) produced | Access |
|---|---|---|
| **UCDP-GED v24** | `civil_war_onset`, `civil_war_incidence`, battle deaths | REST API (ucdpapi.pcr.uu.se) |
| **Powell-Thyne Coup Dataset** | `coup_attempt`, `coup_success` | Direct CSV download |
| **PITF State Failure Problem Set** | `ethwar`, `revwar`, `genocide`, `adverse`, `sftprev` | Direct CSV download |

## What this notebook does
1. Downloads UCDP-GED event data via the UCDP REST API (paginated)
2. Downloads Powell-Thyne coup list
3. Downloads PITF State Failure Problem Set
4. Writes each to ADLS as raw parquet
5. Constructs a combined **country-month label table** for model training

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import time
import requests
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# UCDP GED REST API
UCDP_GED_VERSION  = "24.1"   # update when UCDP releases a new version
UCDP_API_BASE     = f"https://ucdpapi.pcr.uu.se/api/gedevents/{UCDP_GED_VERSION}"
UCDP_PAGE_SIZE    = 1000

# Powell-Thyne coup dataset (Thyne & Powell, UCF — updated annually)
POWELL_THYNE_URL  = (
    "https://www.uky.edu/~clthyn2/coup_data/powell_thyne_coups_final.txt"
)

# PITF State Failure Problem Set (Center for Systemic Peace)
PITF_URL = "https://www.systemicpeace.org/inscr/PITF%20SPSS%20Data%202022.csv"

print(f"Run date    : {RUN_DATE}")
print(f"UCDP GED    : v{UCDP_GED_VERSION}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## 1. UCDP-GED

The UCDP API returns events paginated at up to 1,000 per call.
We pull all three violence types:
- `type_of_violence=1` — state-based (armed conflict between organised actors, ≥1 a government)
- `type_of_violence=2` — non-state conflict
- `type_of_violence=3` — one-sided violence (organised actor vs. civilians)

In [ ]:
def fetch_ucdp_page(page: int, pagesize: int = UCDP_PAGE_SIZE) -> dict:
    params = {"pagesize": pagesize, "page": page}
    resp = requests.get(UCDP_API_BASE, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

def pull_all_ucdp_ged() -> pd.DataFrame:
    all_records = []
    page = 1
    # First call to get total count
    first = fetch_ucdp_page(page=1, pagesize=1)
    total = first.get("TotalCount", 0)
    total_pages = (total // UCDP_PAGE_SIZE) + 1
    print(f"UCDP-GED: {total:,} events across ~{total_pages} pages")

    pbar = tqdm(total=total_pages, desc="UCDP pages", unit="page")
    while True:
        data = fetch_ucdp_page(page)
        records = data.get("Result", [])
        if not records:
            break
        all_records.extend(records)
        pbar.update(1)
        pbar.set_postfix({"total": len(all_records)})
        if len(records) < UCDP_PAGE_SIZE:
            break
        page += 1
        time.sleep(0.2)  # be polite to the UCDP API
    pbar.close()
    return pd.DataFrame(all_records)

print("Pulling UCDP-GED...")
df_ucdp = pull_all_ucdp_ged()
print(f"Total UCDP-GED events: {len(df_ucdp):,}")
df_ucdp.head(2)

In [ ]:
# Type cast key columns
for col in ["year", "type_of_violence", "best", "high", "low",
            "deaths_a", "deaths_b", "deaths_civilians", "deaths_unknown"]:
    if col in df_ucdp.columns:
        df_ucdp[col] = pd.to_numeric(df_ucdp[col], errors="coerce").astype("Int64")

for col in ["latitude", "longitude"]:
    if col in df_ucdp.columns:
        df_ucdp[col] = pd.to_numeric(df_ucdp[col], errors="coerce")

for col in ["date_start", "date_end"]:
    if col in df_ucdp.columns:
        df_ucdp[col] = pd.to_datetime(df_ucdp[col], errors="coerce")

print(f"Years: {df_ucdp['year'].min()}–{df_ucdp['year'].max()}")
print(df_ucdp["type_of_violence"].value_counts().rename({1: 'state-based', 2: 'non-state', 3: 'one-sided'}))

In [ ]:
write_parquet(df_ucdp, f"raw/ucdp_ged/{RUN_DATE}/ucdp_ged_events.parquet")

## 2. Powell-Thyne Coup Dataset

In [ ]:
print(f"Downloading Powell-Thyne from {POWELL_THYNE_URL} ...")
resp = requests.get(POWELL_THYNE_URL, timeout=60)
resp.raise_for_status()

# File is tab-delimited
df_coups = pd.read_csv(io.BytesIO(resp.content), sep="\t", low_memory=False)
print(f"Raw shape: {df_coups.shape}")
print(f"Columns: {list(df_coups.columns)}")
df_coups.head(3)

In [ ]:
# Standardise column names (Powell-Thyne schema varies slightly by release year)
col_map = {
    c: c.lower().strip().replace(" ", "_") for c in df_coups.columns
}
df_coups.rename(columns=col_map, inplace=True)

for col in ["year", "month", "day", "coup"]:
    if col in df_coups.columns:
        df_coups[col] = pd.to_numeric(df_coups[col], errors="coerce").astype("Int64")

# coup == 1 → successful; coup == 2 → attempted/failed
if "coup" in df_coups.columns:
    df_coups["coup_success"]  = (df_coups["coup"] == 1).astype("Int8")
    df_coups["coup_attempt"]  = df_coups["coup"].isin([1, 2]).astype("Int8")

print(f"Coups: {len(df_coups):,} events | {df_coups['year'].min()}–{df_coups['year'].max()}")
print(df_coups["coup"].value_counts().rename({1: 'successful', 2: 'attempted'}))

In [ ]:
write_parquet(df_coups, f"raw/powell_thyne/{RUN_DATE}/powell_thyne_coups.parquet")

## 3. PITF State Failure Problem Set

In [ ]:
print(f"Downloading PITF State Failure data...")
resp = requests.get(PITF_URL, timeout=60)
resp.raise_for_status()

df_pitf = pd.read_csv(io.BytesIO(resp.content), low_memory=False)
print(f"Raw shape: {df_pitf.shape}")
df_pitf.head(3)

In [ ]:
# Standardise column names
df_pitf.columns = [c.lower().strip() for c in df_pitf.columns]

# Key binary onset labels
label_cols = ["sftprev", "ethwar", "revwar", "genocide", "adverse"]
for col in label_cols:
    if col in df_pitf.columns:
        df_pitf[col] = pd.to_numeric(df_pitf[col], errors="coerce").astype("Int8")

if "year" in df_pitf.columns:
    df_pitf["year"] = pd.to_numeric(df_pitf["year"], errors="coerce").astype("Int64")

present_labels = [c for c in label_cols if c in df_pitf.columns]
print(f"PITF shape: {df_pitf.shape}")
print(f"Label columns found: {present_labels}")
if present_labels:
    print(df_pitf[present_labels].sum())

In [ ]:
write_parquet(df_pitf, f"raw/pitf/{RUN_DATE}/pitf_state_failure.parquet")

## Summary

In [ ]:
print("=" * 55)
print("Conflict labels pull complete")
print("=" * 55)
print(f"  UCDP-GED     : {len(df_ucdp):,} events | years {df_ucdp['year'].min()}–{df_ucdp['year'].max()}")
print(f"  Powell-Thyne : {len(df_coups):,} coup events")
print(f"  PITF         : {len(df_pitf):,} country-years")
print()
print("ADLS paths written:")
print(f"  raw/ucdp_ged/{RUN_DATE}/ucdp_ged_events.parquet")
print(f"  raw/powell_thyne/{RUN_DATE}/powell_thyne_coups.parquet")
print(f"  raw/pitf/{RUN_DATE}/pitf_state_failure.parquet")